### Model Training
Nesta etapa treinamos modelos de machine learning para identificar transações potencialmente fraudulentas.

Utilizamos as features criadas anteriormente e aplicamos algoritmos de classificação disponíveis no Spark ML para prever a variável alvo is_fraud.

O objetivo é construir um modelo capaz de identificar padrões associados a fraudes financeiras.

In [0]:
# Imports
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

fraud_df = spark.read.table("fraud_features")

**1. Selecionar features e criar vetor**

In [0]:
# Preparar features para modelagem

# Listar todas as features (exceto transaction_id e is_fraud)
feature_cols = [
    "amount",
    "transaction_hour",
    "foreign_transaction",
    "location_mismatch",
    "device_trust_score",
    "velocity_last_24h",
    "cardholder_age",
    "high_value_transaction",
    "foreign_high_risk",
    "night_transaction",
    "amount_log",
    "merchant_category_index"
]

print(f"Total de features: {len(feature_cols)}")
print("\nFeatures selecionadas:")
for feat in feature_cols:
    print(f"  - {feat}")

missing = [f for f in feature_cols if f not in fraud_df.columns]
if missing:
    print(f"\nFeatures faltando: {missing}")
else:
    print("\nTodas as features estão presentes")

In [0]:
# Criar vetor de features

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

fraud_df = assembler.transform(fraud_df)

print("Vetor de features criado!")
fraud_df.select("features", "is_fraud").show(5, truncate=False)

In [0]:
# Dividir dados em treino e teste

train_df, test_df = fraud_df.randomSplit([0.8, 0.2], seed=42)

print(f"Total de registros: {fraud_df.count()}")
print(f"Treino: {train_df.count()}")
print(f"Teste: {test_df.count()}")

# Verificar distribuição de fraudes
print("\nDistribuição no treino:")
train_df.groupBy("is_fraud").count().show()

print("Distribuição no teste:")
test_df.groupBy("is_fraud").count().show()

In [0]:
# Treinar Logistic Regression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="is_fraud",
    maxIter=10
)

print("Treinando Logistic Regression...")
lr_model = lr.fit(train_df)
print("Modelo treinado!")

# Fazer predições
lr_predictions = lr_model.transform(test_df)

lr_predictions.select("is_fraud", "prediction", "probability").show(10)

In [0]:
# Avaliar Logistic Regression

# Avaliador binário (AUC)
binary_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud",
    metricName="areaUnderROC"
)

lr_auc = binary_evaluator.evaluate(lr_predictions)
print(f"Logistic Regression AUC: {lr_auc:.4f}")

# Avaliador multiclass (Accuracy, Precision, Recall, F1)
multi_evaluator = MulticlassClassificationEvaluator(
    labelCol="is_fraud",
    predictionCol="prediction"
)

lr_accuracy = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "accuracy"})
lr_precision = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "weightedPrecision"})
lr_recall = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "weightedRecall"})
lr_f1 = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "f1"})

print(f"\nLogistic Regression Metrics:")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall: {lr_recall:.4f}")
print(f"F1-Score: {lr_f1:.4f}")

In [0]:
# Matriz de Confusão Logistic Regression

confusion_matrix = lr_predictions.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction")
confusion_matrix.show()

# Calcular métricas manualmente
tp = lr_predictions.filter((col("is_fraud") == 1) & (col("prediction") == 1.0)).count()
tn = lr_predictions.filter((col("is_fraud") == 0) & (col("prediction") == 0.0)).count()
fp = lr_predictions.filter((col("is_fraud") == 0) & (col("prediction") == 1.0)).count()
fn = lr_predictions.filter((col("is_fraud") == 1) & (col("prediction") == 0.0)).count()

print("\nMatriz de Confusão:")
print(f"True Positives: {tp}")
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")

if (tp + fp) > 0:
    precision = tp / (tp + fp)
    print(f"\nPrecision (manual): {precision:.4f}")

if (tp + fn) > 0:
    recall = tp / (tp + fn)
    print(f"Recall (manual): {recall:.4f}")

In [0]:
# Treinar Random Forest

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_fraud",
    numTrees=100,
    maxDepth=10,
    seed=42
)

print("Treinando Random Forest...")
rf_model = rf.fit(train_df)
print("Modelo treinado!")

rf_predictions = rf_model.transform(test_df)

# Avaliar
rf_auc = binary_evaluator.evaluate(rf_predictions)
rf_accuracy = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "accuracy"})
rf_f1 = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "f1"})

print(f"\nRandom Forest Metrics:")
print(f"AUC: {rf_auc:.4f}")
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1-Score: {rf_f1:.4f}")

In [0]:
if rf_auc > lr_auc:
    print("\nMelhor modelo: Random Forest")
    best_model = rf_model
else:
    print("\nMelhor modelo: Logistic Regression")
    best_model = lr_model

In [0]:
# Mostrar contagem de predições
if 'rf_predictions' in dir():
    print(f"\nTotal de predições RF: {rf_predictions.count()}")
    print("Amostra das predições:")
    rf_predictions.select("is_fraud", "prediction", "probability").show(5)
else:
    print("\nNão foi criado")

In [0]:
# Usar mergeSchema
rf_predictions.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("rf_predictions")
print("rf_predictions salvo como tabela permanente")
print(f"Total de registros salvos: {rf_predictions.count()}")